# 01 · Data preparation for GroupBuy
**Purpose:** turn the raw mock survey into a clean customer table, then simulate the Year-1 operating data the deck's business model would produce.

**Data status:** `GroupBuy_Mock_Survey_Data.csv` is *mock* data supplied with the project. Products, suppliers, campaigns and transactions are *synthetic*, generated below. Neither is real market evidence.

In [1]:
import numpy as np, pandas as pd, re
RAW='../Dataset/raw/GroupBuy_Mock_Survey_Data.csv'; OUT='../Dataset/'
SEED=42; rng=np.random.default_rng(SEED)
raw=pd.read_csv(RAW); print('raw shape',raw.shape)

raw shape (1215, 63)


## 1. Cleaning the survey
Each rule is logged so that the changes can be audited. Implausible values become missing rather than being guessed; model pipelines impute them later using **training data only**, which avoids leakage.

In [2]:
df=raw.copy(); log=[]
def note(issue,col,n,action): log.append(dict(issue=issue,column=col,rows_affected=int(n),action=action))
# duplicates: same respondent submitted twice with a different date
n=df.Resp_ID.duplicated().sum(); df=df.drop_duplicates('Resp_ID',keep='first'); note('Duplicate submissions','Resp_ID',n,'Kept first submission')
# spend stored as text such as "AED 1,310"; -1 is an error code
s=df.B1_Monthly_Spend_Eligible_AED.astype(str); bad=~s.str.fullmatch(r'\d+(\.\d+)?')
df['B1_Monthly_Spend_Eligible_AED']=pd.to_numeric(s.str.replace(r'[^0-9.\-]','',regex=True)); note('Text-formatted or error spend','B1_Monthly_Spend_Eligible_AED',bad.sum(),'Parsed text; -1 set missing')
df.loc[df.B1_Monthly_Spend_Eligible_AED<0,'B1_Monthly_Spend_Eligible_AED']=np.nan
# plausible ranges
for c,lo,hi in [('A2_Age',18,80),('A6_Household_Size',1,10),('A7_Monthly_Income_AED',2000,150000),('B2_Online_Orders_Per_Month',0,40),('F1_Min_Discount_To_Switch_Pct',0,60),('G1_Likelihood_Join_Next_3M_0to10',0,10)]:
    m=(df[c]<lo)|(df[c]>hi); df[c]=df[c].astype(float); df.loc[m,c]=np.nan; note('Out-of-range value',c,m.sum(),f'Outside {lo}-{hi} set missing')
# inconsistent labels
def lab(c,f,issue='Inconsistent labels'):
    before=df[c].copy(); df[c]=df[c].map(f,na_action='ignore'); note(issue,c,(before.fillna('~')!=df[c].fillna('~')).sum(),'Standardised')
lab('A3_Gender',lambda x:{'f':'Female','female':'Female','femal':'Female','m':'Male','male':'Male'}.get(x.lower(),x))
amap={'al barsha':'Al Barsha / JVC','jvc':'Al Barsha / JVC','marina':'Dubai Marina / JLT','jlt':'Dubai Marina / JLT','dubai marina':'Dubai Marina / JLT','bur dubai':'Deira / Bur Dubai','deira':'Deira / Bur Dubai','deira/bur dubai':'Deira / Bur Dubai','downtown':'Downtown / Business Bay','business bay':'Downtown / Business Bay'}
lab('A5_Area',lambda x:amap.get(x.lower(),x))
lab('B3_Primary_Channel',lambda x:{'noon':'Noon','noon.com':'Noon','amazon':'Amazon.ae'}.get(x.lower(),x))
lab('G2_Register_For_Pilot',lambda x:{'y':'Yes','yes':'Yes','n':'No','no':'No','maybe':'Maybe','not sure':'Maybe'}[x.lower()])
df['A9_SME_Employees']=df.A9_SME_Employees.fillna(0)  # structural: individuals have no SME staff
df['Is_SME']=(df.A1_Respondent_Type=='Business buyer (SME)').astype(int)
df=df.rename(columns={'Resp_ID':'Customer_ID'})
pd.DataFrame(log).to_csv('../Results/Data_Cleaning_Log.csv',index=False); df.to_csv(OUT+'customers.csv',index=False)
print(pd.DataFrame(log).to_string()); print('clean shape',df.shape); print(df.isna().sum()[lambda s:s>0].to_string())

                            issue                            column  rows_affected                           action
0           Duplicate submissions                           Resp_ID             15            Kept first submission
1   Text-formatted or error spend     B1_Monthly_Spend_Eligible_AED             23      Parsed text; -1 set missing
2              Out-of-range value                            A2_Age              4        Outside 18-80 set missing
3              Out-of-range value                 A6_Household_Size              6         Outside 1-10 set missing
4              Out-of-range value             A7_Monthly_Income_AED             13  Outside 2000-150000 set missing
5              Out-of-range value        B2_Online_Orders_Per_Month              5         Outside 0-40 set missing
6              Out-of-range value     F1_Min_Discount_To_Switch_Pct              6         Outside 0-60 set missing
7              Out-of-range value  G1_Likelihood_Join_Next_3M_0to10     

## 2. Synthetic operating data
The deck describes a process (campaign → pooled demand → supplier bids → final offer → buyer approval → fulfilment) but provides no transactions. We simulate **Year 1 (Jan–Dec 2027)** using the deck's terms: 7% commission, 2% variable cost and AED 15,000 monthly fixed costs. Survey answers drive behaviour:
- campaigns are chosen in proportion to survey category interest;
- only buyers interested in a category commit to it, and order size comes from their stated order value;
- a buyer pays only if the delivered discount clears their stated minimum discount, and is less likely to pay if the campaign exceeds their maximum wait.

Assumptions introduced here (SKU prices, bulk-discount ceilings, 120 campaigns, reliability rates) are the author's, not facts about the startup.

In [3]:
catalog={
'Groceries_Staples':(.14,[('Basmati Rice 10kg','bag',62),('Sunflower Oil 5L','bottle',38),('Sugar 5kg','bag',22),('Lentils 5kg','bag',41)]),
'Household_Cleaning':(.20,[('Laundry Liquid 5L','bottle',48),('Dishwash 5L','bottle',26),('Floor Cleaner 5L','bottle',24),('Bin Liners 100pc','pack',29)]),
'Personal_Care':(.18,[('Shampoo 1L','bottle',32),('Hand Soap 5L','bottle',34),('Toothpaste 6-pack','pack',36),('Body Wash 1L','bottle',28)]),
'Baby_Kids':(.15,[('Diapers Jumbo Pack','pack',89),('Baby Wipes 12-pack','pack',54),('Formula 900g','tin',98),('School Stationery Set','set',46)]),
'Electronics':(.09,[('Wireless Earbuds','unit',179),('Power Bank 20000mAh','unit',109),('24-inch Monitor','unit',499),('Smart Plug 4-pack','pack',129)]),
'Home_Appliances':(.12,[('Air Fryer 5L','unit',299),('Vacuum Cleaner','unit',449),('Water Dispenser','unit',549),('Blender','unit',189)]),
'Furniture_Decor':(.20,[('Office Chair','unit',399),('Bookshelf','unit',259),('Blackout Curtains','set',149),('LED Floor Lamp','unit',119)]),
'Office_Supplies':(.20,[('A4 Paper Box','box',115),('Toner Cartridge','unit',229),('Pens 50-pack','pack',39),('Sticky Notes Bulk','pack',44)]),
'Water_Beverages':(.18,[('Water 5-Gallon x4','bundle',36),('Bottled Water 24-pack','pack',19),('Tea 300 bags','box',42),('Coffee Beans 1kg','bag',85)]),
'Fitness_Sports':(.18,[('Yoga Mat','unit',69),('Dumbbell Set 20kg','set',229),('Whey Protein 2kg','tub',249),('Resistance Bands','set',59)]),
'Fresh_Produce_Meat':(.10,[('Frozen Chicken 10kg','case',139),('Fruit Box 5kg','box',65),('Vegetable Box 5kg','box',49),('Eggs 90-tray','tray',52)]),
'Pet_Supplies':(.16,[('Dry Dog Food 12kg','bag',169),('Cat Litter 15L','bag',59),('Cat Food 24-pack','pack',96),('Pet Pads 100pc','pack',79)])}
rows=[]
for cat,(ceil,items) in catalog.items():
    for name,unit,p in items: rows.append(dict(Product_ID=f'P{len(rows)+1:03d}',Category=cat,Product_Name=name,Unit=unit,Reference_Price_AED=p,Bulk_Discount_Ceiling=ceil))
products=pd.DataFrame(rows); products.to_csv(OUT+'products.csv',index=False)
cats=list(catalog)
sup=[]
for i in range(1,25):
    sup.append(dict(Supplier_ID=f'S{i:02d}',Supplier_Name=f'Supplier {i:02d} (fictional)',Categories='|'.join(rng.choice(cats,4,replace=False)),Reliability=round(float(rng.beta(16,3)),3),Price_Competitiveness=round(float(rng.normal(0,.012)),4)))
suppliers=pd.DataFrame(sup); suppliers.to_csv(OUT+'suppliers.csv',index=False)
print(len(products),'products;',len(suppliers),'suppliers')

48 products; 24 suppliers


In [4]:
cust=df.copy()
cust['pscore']=cust.Conversion_Score_0to10.clip(0,10)
cust['F1']=cust.F1_Min_Discount_To_Switch_Pct.fillna(cust.F1_Min_Discount_To_Switch_Pct.median())
cust['F2']=cust.F2_Max_Wait_Days.fillna(cust.F2_Max_Wait_Days.median())
interest=np.array([cust[f'CAT_{c}'].sum() for c in cats],float); interest/=interest.sum()
month_n=np.linspace(4,16,12).round().astype(int)
camps=[];txns=[];cid=0;tid=0
for mi,nm in enumerate(month_n,1):
    for _ in range(nm):
        cid+=1; cat=rng.choice(cats,p=interest); ceil,_=catalog[cat]
        p=products[products.Category==cat].sample(1,random_state=int(rng.integers(1e9))).iloc[0]; ref=p.Reference_Price_AED
        start=pd.Timestamp(2027,mi,int(rng.integers(1,26))); dur=int(rng.integers(4,15))
        target_val=float(np.exp(rng.normal(np.log(20000),.45))); target_q=int(max(10,round(target_val/ref)))
        pool=cust[(cust[f'CAT_{cat}']==1)&(cust.G4_Expected_Order_Value_AED*2.5>=ref)]
        n=min(len(pool),int(rng.poisson(6+1.4*mi+.0009*target_val)))
        w=(pool.pscore+.5).values; buyers=pool.iloc[rng.choice(len(pool),n,replace=False,p=w/w.sum())]
        share=np.clip(rng.lognormal(np.log(.8),.35,n),.2,2)
        q=np.maximum(1,np.round(buyers.G4_Expected_Order_Value_AED.values*share/ref)).astype(int)
        pledged=int(q.sum()); pv=pledged*ref
        nb=int(np.clip(rng.poisson(1.5+pv/9000),1,8)); sids=suppliers[suppliers.Categories.str.contains(cat)].Supplier_ID.tolist() or suppliers.Supplier_ID.tolist()
        sid=rng.choice(sids)
        comp=float(suppliers.loc[suppliers.Supplier_ID==sid,'Price_Competitiveness'].iloc[0])
        gross=np.clip(ceil*.55+.022*np.log1p(pv/5000)+.006*nb+comp+rng.normal(0,.018),.01,ceil+.04)
        delivery=rng.uniform(.012,.04)   # delivery as a share of reference price
        net=gross-delivery; delivered=round(ref*(1-net),2)
        status='Offer made'
        if pledged<.7*target_q: status='Cancelled - target not met'
        elif net<.05: status='Cancelled - price not acceptable'
        paid=np.zeros(n,bool)
        if status=='Offer made':
            gap=(net*100-buyers.F1.values)/6; late=(dur>buyers.F2.values)*-.7
            pr=1/(1+np.exp(-(gap+late+.22*(buyers.pscore.values-5)+.5)))
            paid=rng.random(n)<pr
            if q[paid].sum()<.4*target_q: status='Cancelled - low conversion'; paid[:]=False
            else: status='Completed'
        rel=float(suppliers.loc[suppliers.Supplier_ID==sid,'Reliability'].iloc[0])
        ontime=bool(rng.random()<rel) if status=='Completed' else None
        for j,(b,qq) in enumerate(zip(buyers.Customer_ID,q)):
            tid+=1; pd_=bool(paid[j]); val=round(qq*delivered,2) if pd_ else 0.
            late_order=(status=='Completed') and (ontime is False)
            dis=bool(pd_ and rng.random()<(.14 if late_order else .025))
            txns.append(dict(Transaction_ID=f'T{tid:05d}',Campaign_ID=f'C{cid:04d}',Customer_ID=b,Campaign_Start=start.date(),Category=cat,Product_ID=p.Product_ID,Quantity=int(qq),Committed_Value_AED=round(qq*ref,2),Paid=int(pd_),GMV_AED=val,Savings_AED=round(qq*(ref-delivered),2) if pd_ else 0.,Commission_AED=round(val*.07,2),Variable_Cost_AED=round(val*.02,2),Dispute=int(dis)))
        camps.append(dict(Campaign_ID=f'C{cid:04d}',Start_Date=start.date(),Duration_Days=dur,Category=cat,Product_ID=p.Product_ID,Supplier_ID=sid,Reference_Price_AED=ref,Target_Qty=target_q,Pledged_Qty=pledged,Committed_Buyers=n,Supplier_Bids=nb,Delivered_Price_AED=delivered,Net_Discount_Pct=round(net*100,2),Status=status,Paid_Buyers=int(paid.sum()),Paid_Qty=int(q[paid].sum()),GMV_AED=round(float((q[paid]*delivered).sum()),2),On_Time=(None if ontime is None else int(ontime))))
campaigns=pd.DataFrame(camps); tx=pd.DataFrame(txns)
campaigns.to_csv(OUT+'campaigns.csv',index=False); tx.to_csv(OUT+'transactions.csv',index=False)
print(campaigns.Status.value_counts().to_string()); print(campaigns.Net_Discount_Pct.describe().round(1).to_string()); print('commitments',len(tx),'paid',tx.Paid.sum(),'GMV',tx.GMV_AED.sum().round(),'commission',tx.Commission_AED.sum().round())

Status
Completed                           97
Cancelled - low conversion          10
Cancelled - target not met          10
Cancelled - price not acceptable     3
count    120.0
mean      13.1
std        3.9
min        3.9
25%       10.2
50%       13.0
75%       15.9
max       22.6
commitments 4449 paid 2218 GMV 1861877.0 commission 130331.0


## 3. Integrity checks

In [5]:
assert tx.Customer_ID.isin(df.Customer_ID).all() and tx.Campaign_ID.isin(campaigns.Campaign_ID).all() and campaigns.Product_ID.isin(products.Product_ID).all() and campaigns.Supplier_ID.isin(suppliers.Supplier_ID).all()
assert abs(tx.GMV_AED.sum()-campaigns.GMV_AED.sum())<5
print('All foreign keys resolve; transaction and campaign GMV reconcile.')

All foreign keys resolve; transaction and campaign GMV reconcile.
